# Workshop Databricks — Automotivo
### Guia Passo a Passo

Bem-vindo(a)! Neste workshop você vai aprender a usar a Databricks para **explorar, transformar e analisar dados**.

**Agenda do dia:**

| # | Tópico | Obrigatório? | O que você vai fazer |
|---|---|---|---|
| 1 | Upload de arquivos (CSV) | Sim | Subir CSVs e criar tabelas |
| 2 | XLSX + Genie Code + Notebook | Opcional | Transformar Excel e criar tabelas |
| 3 | Lakeflow Designer | Opcional | Pipeline visual de dados |
| 4 | AI Functions | Opcional | IA dentro do SQL |
| 5 | Genie Agents | Sim | Perguntas em linguagem natural (modo Agent) |
| 6 | Genie One | Sim | Explorar + agendar report diário |
| 7 | AI/BI Dashboards | Sim | Dashboard com 2 páginas |


## Descrição das tabelas

### Tabela `clientes`
Cadastro de clientes do braço financeiro (financiamento, CDC, leasing e consórcio).

| Coluna | Descrição |
|---|---|
| `cliente_id` | Identificador único do cliente |
| `nome` | Nome completo do cliente |
| `cpf_mascarado` | CPF parcialmente oculto para privacidade (ex.: `***.132.***-**`) |
| `idade` | Idade do cliente em anos |
| `renda_mensal` | Renda mensal declarada em R$ |
| `score_credito` | Score de crédito (faixa típica 300–950) |
| `cidade` | Cidade de residência |
| `estado` | UF da residência |
| `segmento` | Perfil do cliente: `PF`, `PJ Frota` ou `Consorciado` |
| `canal_aquisicao` | Canal de origem: `Concessionária`, `Digital`, `Telefone` ou `Parceiro` |
| `data_cadastro` | Data em que o cliente foi cadastrado |
| `ativo` | Indica se o cliente está ativo (`true` / `false`) |

---

### Tabela `contratos`
Contratos e propostas de crédito vinculados aos clientes (produto, veículo, grupo e desfecho).

| Coluna | Descrição |
|---|---|
| `contrato_id` | Identificador único do contrato |
| `cliente_id` | Referência ao cliente (`clientes.cliente_id`) |
| `proposta_id` | Identificador da proposta comercial/crédito |
| `grupo_consorcio` | Código do grupo de consórcio (ex.: `G001`); vazio se não for consórcio |
| `tipo_produto` | Produto contratado: `Financiamento Veículo`, `Consórcio`, `CDC` ou `Leasing` |
| `modelo_veiculo` | Modelo do veículo |
| `categoria_veiculo` | Categoria do veículo: `Passeio` ou `SUV` |
| `valor_financiado` | Valor financiado em R$ |
| `entrada_pct` | Percentual de entrada sobre o valor (ex.: `0.20` = 20%) |
| `prazo_meses` | Prazo do contrato em meses |
| `taxa_juros_am` | Taxa de juros ao mês (%) |
| `status_contrato` | Situação atual: `Ativo`, `Quitado`, `Cancelado` ou `Inadimplente` |
| `motivo_cancelamento` | Motivo do cancelamento (quando `status_contrato = Cancelado`) |
| `data_contratacao` | Data de contratação / início do contrato |
| `data_status` | Data da última atualização do status |
| `aprovado` | Indica se a proposta foi aprovada (`true` / `false`) |
| `sucesso` | Indica desfecho positivo (`true` para Ativo/Quitado; `false` para Cancelado/Inadimplente) |

---

### Tabela `parcelas_cobranca`
Parcelas dos contratos, com inadimplência, etapas do processo de cobrança e potencial de recuperação.

| Coluna | Descrição |
|---|---|
| `parcela_id` | Identificador único da parcela |
| `contrato_id` | Referência ao contrato (`contratos.contrato_id`) |
| `cliente_id` | Referência ao cliente (`clientes.cliente_id`) |
| `numero_parcela` | Número sequencial da parcela no contrato |
| `data_vencimento` | Data de vencimento da parcela |
| `valor_parcela` | Valor da parcela em R$ |
| `status_parcela` | Situação do pagamento: `Paga`, `Pendente`, `Vencida` ou `Renegociada` |
| `dias_atraso` | Quantidade de dias em atraso (0 se em dia) |
| `etapa_cobranca` | Etapa atual do processo: `Sem Cobrança`, `SMS`, `Email`, `Ligação`, `Cartório` ou `Jurídico` |
| `data_pagamento` | Data em que a parcela foi paga/recuperada (vazio se ainda em aberto) |
| `valor_recuperado` | Valor efetivamente recuperado em R$ |
| `potencial_recuperacao` | Classificação de recuperação: `Alto`, `Médio` ou `Baixo` |
| `agente_cobranca` | Equipe/agente responsável pela cobrança |
| `mes` | Mês de vencimento da parcela (1–12) |
| `ano` | Ano de vencimento da parcela |

---

### Tabela `comunicacoes`
Histórico e fila de comunicações (enviadas e agendadas) para cobrança, oferta, CNH e entrega.

| Coluna | Descrição |
|---|---|
| `comunicacao_id` | Identificador único da comunicação |
| `cliente_id` | Referência ao cliente (`clientes.cliente_id`) |
| `contrato_id` | Referência ao contrato (`contratos.contrato_id`) |
| `tipo_comunicacao` | Canal de envio: `SMS`, `Email`, `WhatsApp`, `Carta` ou `Push` |
| `campanha` | Tipo de campanha: `Cobrança`, `Renovação`, `Oferta Cross-sell`, `Lembrete Parcela`, `CNH Pendência` ou `Entrega Veículo` |
| `data_prevista_envio` | Data prevista de envio (inclui datas futuras para previsão) |
| `data_envio` | Data efetiva do envio (vazio se ainda não enviado) |
| `status_envio` | Status do disparo: `Enviado`, `Agendado`, `Falhou` ou `Cancelado` |
| `resultado` | Resultado após envio: `Respondido`, `Sem Resposta`, `Pagou` ou `Opt-out` |
| `mes_referencia` | Mês da data prevista de envio |
| `ano_referencia` | Ano da data prevista de envio |
| `custo_estimado_rs` | Custo unitário estimado da comunicação em R$ |

---

### Arquivo opcional `jornada_cnh_mercado.xlsx`

#### Aba `jornada_cnh`
Pendências e entregas da jornada CNH. Colunas com nomes “sujos” e datas em `dd/mm/yyyy` de propósito (para exercício de Genie Code).

| Coluna | Descrição |
|---|---|
| `ID Jornada` | Identificador da jornada CNH |
| `ID Cliente` | Referência ao cliente |
| `ID Contrato` | Referência ao contrato |
| `Status CNH` | Status atual: `Entregue`, `Pendente Documentação`, `Pendente Exame` ou `Em Análise` |
| `Tipo Pendencia` | Motivo da pendência (ex.: documento ilegível, CPF divergente) ou `Sem pendência` |
| `Data Solicitacao` | Data da solicitação (formato `dd/mm/yyyy`) |
| `Data Prevista Entrega` | Data prevista de entrega (formato `dd/mm/yyyy`) |
| `Data Entrega Real` | Data real de entrega (formato `dd/mm/yyyy`; vazio se não entregue) |
| `Dias Atraso Entrega` | Dias de atraso em relação à data prevista |
| `Etapa Jornada` | Etapa atual: `Cadastro`, `Documentação`, `Análise`, `Produção CNH`, `Expedição` ou `Entrega` |
| `Canal Entrega` | Canal: `Correios`, `Retirada Concessionária`, `Motoboy` ou `Digital` |
| `Flag Pendencia` | Indica pendência: `SIM` / `NAO` |
| `Entrega No Prazo` | Indica entrega no prazo: `SIM` / `NAO` (vazio se ainda não entregue) |
| `Score Experiencia` | Nota de experiência do cliente na jornada (0–100) |
| `Observacao` | Comentário livre operacional |

#### Aba `mercado_concorrentes`
Série mensal de volume e indicadores dos agentes financeiros concorrentes (para sazonalidade e share).

| Coluna | Descrição |
|---|---|
| `mercado_id` | Identificador do registro mensal |
| `ano` | Ano de referência |
| `mes` | Mês de referência (1–12) |
| `agente_financeiro` | nome do agente financeiro |
| `volume_contratos` | Volume de contratos no mês |
| `ticket_medio_rs` | Ticket médio dos contratos em R$ |
| `taxa_media_am` | Taxa média ao mês praticada (%) |
| `taxa_selic_ref` | Taxa Selic de referência no período (%) |
| `ipca_ref` | IPCA de referência no período (%) |
| `participacao_mercado_pct` | Participação de mercado estimada (%) |
| `campanha_ativa` | Campanha vigente no mês (ex.: `Juros zero`, `Entrada facilitada`) |


---
## Configuração inicial — widgets

Preencha os widgets com o catálogo, schema, seu prefixo único e (se usar) o caminho do Volume.


In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("nome_catalogo", "", "Catálogo")
dbutils.widgets.text("nome_schema", "", "Schema")
dbutils.widgets.text("seu_prefixo", "", "Seu Prefixo (ex: anaj_)")
dbutils.widgets.text("caminho_volume", "", "Caminho do Volume (opcional)")


In [0]:
nome_catalogo  = dbutils.widgets.get("nome_catalogo")
nome_schema    = dbutils.widgets.get("nome_schema")
seu_prefixo    = dbutils.widgets.get("seu_prefixo")
caminho_volume = dbutils.widgets.get("caminho_volume")

print(f"Catálogo : {nome_catalogo}")
print(f"Schema   : {nome_schema}")
print(f"Prefixo  : {seu_prefixo}")
print(f"Volume   : {caminho_volume}")


---
## Módulo 1 — Subindo arquivos manualmente (CSV)

### O que é?
O Databricks permite fazer upload de arquivos (CSV, XLS, JSON) pela interface e convertê-los em tabelas Delta no Unity Catalog — sem código.

### Passo a passo

1. No menu lateral, clique em **"+ New"** → **"Add or upload data"** (ou **Catalog** → **Create** → **Create table**)
2. Arraste o CSV ou selecione do computador
3. Revise o schema detectado (tipos das colunas)
4. Escolha **Catálogo**, **Schema** e nome da tabela com o seu prefixo ex: anaj_clientes
5. Clique em **"Create table"**

### Arquivos obrigatórios

| Arquivo | Nome sugerido da tabela |
|---|---|
| `clientes.csv` | `{seu_prefixo}clientes` |
| `contratos.csv` | `{seu_prefixo}contratos` |
| `parcelas_cobranca.csv` | `{seu_prefixo}parcelas_cobranca` |
| `comunicacoes.csv` | `{seu_prefixo}comunicacoes` |

> Exemplo: se seu prefixo for `anaj_`, a tabela fica `anaj_clientes`.

### Validação rápida

Depois do upload, confira no **Catalog** se as 4 tabelas aparecem e abra **Sample data**.


In [0]:
%sql
-- Ajuste o nome se necessário e valide uma amostra
SELECT * FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}clientes` LIMIT 10;


In [0]:
%sql
SELECT status_contrato, COUNT(*) AS qtd
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}contratos`
GROUP BY status_contrato
ORDER BY qtd DESC;


---
## Módulo 2 — XLSX com Genie Code + Notebook *(opcional)*

Objetivo: subir o Excel `jornada_cnh_mercado.xlsx`, limpar/transformar com **Genie Code** (ou notebook) e criar as tabelas `jornada_cnh` e `mercado_concorrentes`.

### 2.1 — Subir o arquivo para o Workspace ou Volume

**Opção A — Volume (recomendado para dados):**
1. **Catalog** → seu schema → **Volumes** → crie `arquivos` se não existir
2. Faça upload de `jornada_cnh_mercado.xlsx` para o Volume
3. Anote o caminho, ex.: `/Volumes/<catalogo>/<schema>/arquivos/jornada_cnh_mercado.xlsx`

**Opção B — Workspace Files:**
1. No navegador de arquivos do workspace, faça upload do XLSX
2. Anote o caminho do arquivo no workspace

### 2.2a — Criar tabela bruta a partir do Excel usando expressão de SQL

No **SQL Editor** ou neste notebook:


In [0]:
%sql
-- Exemplo: criar tabela bruta da aba jornada (ajuste LOCATION)
-- CREATE TABLE ${nome_catalogo}.${nome_schema}.${seu_prefixo}jornada_cnh_raw
-- USING EXCEL
-- OPTIONS (header = 'true', inferSchema = 'true')
-- LOCATION '/Volumes/SEU_CATALOGO/SEU_SCHEMA/arquivos/jornada_cnh_mercado.xlsx';

SELECT 'Substitua o LOCATION pelo caminho real do seu XLSX' AS instrucao;


### 2.2b — Criar tabela bruta a partir do Excel manualmente


1. No menu lateral, clique em **"+ New"** → **"Add or upload data"** (ou **Catalog** → **Create** → **Create table**).
2. Faça upload do arquivo Excel (`.xlsx`) do seu computador.
3. O Databricks detecta as abas (sheets) do Excel: cada aba pode ser materializada como uma tabela separada.
4. Revise o schema detectado para cada aba (tipos das colunas).
5. Escolha o catálogo, schema e nome da tabela (exemplo: `{seu_prefixo}jornada_cnh` para a aba `jornada_cnh`).
6. Repita para cada aba relevante do Excel (exemplo: `jornada_cnh`, `mercado_concorrentes`).
7. Clique em **"Create table"** para cada aba.

> Cada sheet do Excel representa uma tabela distinta. Nomeie cada tabela conforme a aba para facilitar a análise.

### 2.2c Subindo excel manualmente com Genie Code a partir do path ou upload no Genie code

Você pode pedir ao Genie Code para criar a tabela a partir do Excel de duas formas:

- **Usando o caminho de um Volume:** Informe o path do arquivo `.xlsx` já carregado em um Volume (exemplo: `/Volumes/<catalogo>/<schema>/arquivos/jornada_cnh_mercado.xlsx`). O Genie Code irá ler o arquivo diretamente do caminho e criar a tabela.

- **Fazendo upload direto no Genie Code:** No próprio Genie Code, faça upload do Excel do seu computador. O assistente detecta as abas e permite criar a tabela a partir do arquivo recém-enviado.

Em ambos os casos, basta pedir para o Genie Code criar a tabela a partir do Excel, indicando o caminho ou fazendo o upload.

### 2.3a — Transformar o Excel com Genie Code

Abra um novo notebook e use o assistente **Genie Code** (✨) com prompts como:

**Prompt 1 — limpar jornada CNH**
```
Use o arquivo jornada_cnh_mercado.xlsx

1- Renomeie as colunas para em português sem espaços:
  id_jornada, cliente_id, contrato_id, status_cnh, tipo_pendencia,
  data_solicitacao, data_prevista_entrega, data_entrega_real,
  dias_atraso_entrega, etapa_jornada, canal_entrega,
  flag_pendencia, entrega_no_prazo, score_experiencia, observacao.
2- Converta as datas de dd/mm/yyyy para DATE.
3- Salve o resultado como tabela Delta
{catalogo}.{schema}.{seu_prefixo}_jornada_cnh_transformado
```

### 2.4 — Alternativa em SQL/Python (sem Genie Code)

Se preferir, use SQL com `to_date` e `CASE` após criar a tabela raw, ou um trecho PySpark no notebook para parsear datas BR e gravar em Delta.




---
## Módulo 3 — Lakeflow Designer *(opcional)*

### O que é?
Ferramenta **visual (drag-and-drop)** para pipelines de transformação — ideal para times de negócio.

**Como acessar:** **"+ New"** → **"Visual data prep"** / Lakeflow Designer

### Usando Genie Code no Designer
1. Clique no assistente de IA (✨)
2. Cole as instruções abaixo
3. Revise os nós gerados e confirme

### Instruções sugeridas (Genie Code)

```
Use o catálogo: {seu_catalogo} e schema {seu_schema}

- Faça um join entre {seu_prefixo}parcelas_cobranca e {seu_prefixo}clientes usando cliente_id.
- Faça também join com {seu_prefixo}contratos usando contrato_id.
- Filtre apenas parcelas com status_parcela em ('Vencida', 'Pendente', 'Renegociada').
- Agrupe por estado, segmento, etapa_cobranca e potencial_recuperacao.
- Some valor_parcela e valor_recuperado, calcule média de dias_atraso e conte parcelas.
- Salve o resultado em {catalogo}.{schema}.{seu_prefixo}inadimplencia_consolidada
```

### Passo a passo manual (se preferir)
1. **Add source** → tabelas `clientes`, `contratos`, `parcelas_cobranca`
2. **Join** parcelas ↔ clientes (`cliente_id`) e ↔ contratos (`contrato_id`)
3. **Filter** status em atraso/pendente/renegociada
4. **Aggregate** por estado / segmento / etapa
5. **Save as table** consolidada


---
## Módulo 4 — AI Functions *(opcional)*

Use IA **dentro do SQL** no SQL Editor ou notebook.

### Query — `ai_gen`: resumo de risco do cliente


In [0]:
%sql
SELECT
  c.cliente_id,
  c.nome,
  c.score_credito,
  c.segmento,
  ROUND(SUM(CASE WHEN p.status_parcela IN ('Vencida','Pendente') THEN p.valor_parcela ELSE 0 END), 2) AS valor_em_atraso,
  MAX(p.dias_atraso) AS max_dias_atraso,
  ai_gen(
    CONCAT(
      'Cliente ', c.nome, ' (segmento ', c.segmento, ', score ', c.score_credito,
      '). Valor em atraso: R$ ',
      ROUND(SUM(CASE WHEN p.status_parcela IN ('Vencida','Pendente') THEN p.valor_parcela ELSE 0 END), 2),
      '. Máximo de dias em atraso: ', MAX(p.dias_atraso),
      '. Escreva em português um resumo executivo de 2 frases com risco e próxima ação de cobrança.'
    )
  ) AS resumo_ia
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}clientes` c
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}parcelas_cobranca` p
  ON c.cliente_id = p.cliente_id
GROUP BY c.cliente_id, c.nome, c.score_credito, c.segmento
HAVING valor_em_atraso > 0
ORDER BY valor_em_atraso DESC
LIMIT 5;


### Query — `ai_classify`: prioridade de cobrança


In [0]:
%sql
SELECT
  parcela_id,
  cliente_id,
  status_parcela,
  dias_atraso,
  etapa_cobranca,
  potencial_recuperacao,
  valor_parcela,
  ai_classify(
    CONCAT(
      'Status: ', status_parcela,
      '. Dias atraso: ', dias_atraso,
      '. Etapa: ', etapa_cobranca,
      '. Potencial recuperação: ', potencial_recuperacao,
      '. Valor: ', valor_parcela
    ),
    ARRAY('Prioridade Alta', 'Prioridade Média', 'Prioridade Baixa')
  ) AS prioridade_ia
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}parcelas_cobranca`
WHERE status_parcela IN ('Vencida', 'Pendente')
LIMIT 20;


---
## Módulo 5 — Genie Agents *(obrigatório)*

### O que é?
O **Genie** é o assistente de BI conversacional da Databricks. No modo **Agent**, ele faz análises mais profundas, multi-tabela e hipotéticas.

1. Abra o menu **Genie Agents** no Databricks.
2. Crie ou selecione um Agent.
3. Adicione as tabelas necessárias ao Space:
   - `clientes`
   - `contratos`
   - `parcelas_cobranca`
   - `comunicacoes`
   - (Opcional) `jornada_cnh` e `mercado_concorrentes` se você criou essas tabelas.
4. Ative o modo **Agent** para realizar análises avançadas.
**Instrução para o Genie Agent**:  
```
Você é um assistente de BI especializado em inadimplência, recuperação de crédito e análise de processos financeiros. 
 
Siga as regras abaixo ao responder:

1. Sempre tente utilizar gráficos para explicar as métricas e dados.
2. Destaque insights operacionais se houver: gargalos do processo, oportunidades de recuperação e recomendações de ação.

```
---

### 5.1 — Perguntas alinhadas aos temas do time

**Inadimplência e eficiência de processos**

Quais fatores influenciam a inadimplência? Considere score de crédito, renda, segmento, entrada_pct, prazo e tipo de produto.


Onde estão os maiores gargalos do processo de cobrança? Mostre volume e valor por etapa_cobranca.


Qual combinação de etapa_cobranca e agente_cobranca tem pior taxa de recuperação?


**Previsão de comunicações**

Quantas comunicações estão agendadas para os próximos meses? Quebre por mês, tipo_comunicacao e campanha.


Qual o custo estimado total das comunicações agendadas por canal?


**Sucesso vs cancelamento (clientes, propostas e grupos)**

Quais características dos clientes, propostas e grupos estão mais associadas ao sucesso ou cancelamento dos contratos?


Nos consórcios, quais grupos_consorcio têm maior taxa de cancelamento?


Compare taxa de sucesso por canal_aquisicao e por categoria_veiculo.


**Potencial de recuperação**

Quais clientes apresentam maior potencial de recuperação? Liste top 20 por potencial_recuperacao Alto e valor em atraso.


Qual o valor total recuperável se priorizarmos apenas potencial Alto e Médio com menos de 60 dias de atraso?


**Jornada CNH** *(se tiver criado a tabela opcional)*

Quais tipos de pendência CNH mais atrasam a entrega? Qual o impacto no score de experiência?


Quais dados da jornada CNH devemos priorizar nas análises para melhorar indicadores de Pendência CNH e Entregas CNH?


**Mercado / concorrentes** *(se tiver criado a tabela opcional)*

Qual é a sazonalidade dos principais agentes financeiros concorrentes da Honda e o que influencia essa flutuação (Selic, IPCA, campanhas)?


Em quais meses o Banco Honda ganha ou perde participação de mercado?


---

### 5.2 — Mais perguntas para explorar (Genie Agents)


Qual a taxa de inadimplência por estado e por segmento?


Existe correlação entre resultado da comunicação (Pagou) e redução de dias_atraso nas parcelas seguintes?


Quais modelos de veículo concentram mais contratos inadimplentes?


Monte um ranking de produtividade operacional das equipes de cobrança: valor recuperado / quantidade de parcelas tratadas.


Se aumentarmos em 20% as comunicações WhatsApp de cobrança no próximo trimestre, qual o volume adicional e o custo estimado?


Quais clientes ativos com score abaixo de 600 ainda não receberam comunicação de cobrança nos últimos 60 dias?


Crie um perfil típico do cliente que cancela por 'Negado crédito' versus o que cancela por 'Desistência cliente'.


Qual o prazo médio até a primeira parcela vencida após a contratação, por tipo_produto?


---

### 5.3 — Perguntas hipotéticas (Agent / Deep Research)


Se migrarmos todos os clientes com potencial Alto de recuperação para a etapa Ligação em até 15 dias, qual o impacto estimado no valor recuperado?


Se reduzirmos a taxa média de entrada mínima de 20% para 15%, como isso poderia afetar a taxa de cancelamento e inadimplência com base no histórico?


Com base na sazonalidade de comunicações e de inadimplência, em quais meses devemos reforçar headcount de cobrança?

---
## Módulo 6 — Genie One + agendar report diário

### O que é o Genie One?
O **Genie One** / experiência **Databricks One** concentra, em um ambiente mais simples para negócio:

| Capacidade | Uso típico |
|---|---|
| Perguntas em linguagem natural | Análises do dia a dia sem SQL |
| Dashboards / insights | Acompanhar KPIs |
| Reports agendados | Enviar resumo diário/semanal para o time |
| Descoberta de dados | Encontrar tabelas governadas no Unity Catalog |

### 6.1 — Explorar no Genie One
1. Abra o **Genie One** / Databricks One no workspace
2. Selecione o Space / dados do workshop
3. Faça 2–3 perguntas de negócio, por exemplo:
```
Resumo diário: valor em atraso, % inadimplência e top 5 etapas com mais gargalo.
```
```
Quantas comunicações estão agendadas para os próximos 30 dias?
```

### 6.2 — Agendar um report diário

Objetivo: o time de cobrança/risco receber todo dia um resumo automático.

**Passo a passo (UI Genie / Genie One):**
1. Faça uma pergunta boa de report, por exemplo:
```
Gere um relatório diário de inadimplência: total em atraso (R$), quantidade de parcelas vencidas, distribuição por etapa_cobranca, top 10 clientes com maior potencial de recuperação Alto, e volume de comunicações agendadas para hoje e amanhã.
```
2. Após a resposta, procure a opção **Schedule** / **Schedule report** / **Subscribe** (ícone de relógio ou menu ⋮)
3. Configure:
   - **Frequência:** Daily (diário)
   - **Horário:** ex. 08:00 (fuso do workspace)
   - **Destinatários:** e-mail do seu time (cobrança / risco)
   - **Formato:** tabelas + insights da resposta
4. Salve e confirme que o schedule aparece como **Active**

**Critério de aceite deste módulo:**
- [ ] Pergunta de report executada com sucesso
- [ ] Schedule diário criado
- [ ] Pelo menos 1 destinatário configurado

> Se a opção de schedule estiver em **Dashboards** no seu workspace: crie um AI/BI Dashboard simples com os KPIs acima e use **Schedule** → **Email** diário — o objetivo pedagógico é o mesmo (report automático diário).


---
## Módulo 7 — AI/BI Dashboards (2 páginas)

### O que é?
AI/BI Dashboards permitem criar visualizações com linguagem natural e explorar com chat.

**Como acessar:** **Dashboards** → **New dashboard**

### 7.1 — Prompt para criar o dashboard (apenas 2 páginas)

Cole no Genie Code / assistente do dashboard:

```
Crie um dashboard chamado "Painel Serviços Financeiros Auto" com 2 páginas:

Página 1 - "Inadimplência e Cobrança":
- KPIs: valor total em atraso, % de parcelas vencidas, ticket médio em atraso, valor recuperado
- Barras: valor em atraso por etapa_cobranca (gargalos do processo)
- Barras ou tabela: inadimplência por estado e por segmento
- Ranking: top clientes por potencial_recuperacao Alto e valor em atraso
- Filtros: estado, segmento, potencial_recuperacao, período (mes/ano)

Página 2 - "Comunicações, Contratos e Experiência":
- KPIs: comunicações agendadas (próximos meses), custo estimado, taxa de sucesso dos contratos, taxa de cancelamento
- Linha/barras: volume de comunicações por mês (enviadas vs agendadas) e por campanha
- Sucesso vs cancelamento por canal_aquisicao e tipo_produto
- (Se existir) indicadores de pendência CNH e entregas no prazo; sazonalidade dos concorrentes (volume por mês)
- Filtros: campanha, tipo_comunicacao, tipo_produto

Use visualizações claras, tema profissional.
```

### 7.2 — Gráficos manuais (se preferir)

**Página 1**
```
Valor em atraso por etapa_cobranca
```
```
Taxa de inadimplência por segmento
```
```
Top 10 clientes com maior valor em atraso e potencial Alto
```

**Página 2**
```
Comunicações agendadas por mês e tipo_comunicacao
```
```
Taxa de sucesso e cancelamento por canal_aquisicao
```
```
Evolução mensal do volume de contratos dos concorrentes (se tabela existir)
```

### 7.3 — Explorar com o chat do dashboard
```
Onde está o maior gargalo de cobrança agora?
```
```
Quais campanhas de comunicação têm melhor taxa de resultado Pagou?
```
```
Se filtrarmos só Consorciado, como muda a inadimplência?
```

### 7.4 — Compartilhar
1. **Share** → adicione o time
2. Permissão **Can view** ou **Can edit**
3. (Opcional) combine com o schedule diário do Módulo 6


---
## Consultas SQL de apoio (opcional)

Use no SQL Editor para validar os dados antes do Genie.


In [0]:
%sql
-- Gargalos do processo de cobrança
SELECT
  etapa_cobranca,
  COUNT(*) AS qtd_parcelas,
  ROUND(SUM(valor_parcela), 2) AS valor_total,
  ROUND(AVG(dias_atraso), 1) AS media_dias_atraso,
  ROUND(SUM(valor_recuperado), 2) AS valor_recuperado
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}parcelas_cobranca`
WHERE status_parcela IN ('Vencida', 'Pendente', 'Renegociada')
GROUP BY etapa_cobranca
ORDER BY valor_total DESC;


In [0]:
%sql
-- Previsão de comunicações (agendadas)
SELECT
  ano_referencia,
  mes_referencia,
  tipo_comunicacao,
  campanha,
  COUNT(*) AS qtd_agendada,
  ROUND(SUM(custo_estimado_rs), 2) AS custo_estimado
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}comunicacoes`
WHERE status_envio = 'Agendado'
GROUP BY ano_referencia, mes_referencia, tipo_comunicacao, campanha
ORDER BY ano_referencia, mes_referencia, qtd_agendada DESC;


In [0]:
%sql
-- Fatores associados a sucesso vs cancelamento
SELECT
  c.segmento,
  c.canal_aquisicao,
  ct.tipo_produto,
  CASE WHEN ct.entrada_pct < 0.2 THEN 'Entrada < 20%' ELSE 'Entrada >= 20%' END AS faixa_entrada,
  CASE WHEN c.score_credito < 600 THEN 'Score < 600'
       WHEN c.score_credito < 750 THEN 'Score 600-749'
       ELSE 'Score >= 750' END AS faixa_score,
  COUNT(*) AS qtd_contratos,
  ROUND(AVG(CASE WHEN ct.sucesso = true THEN 1 ELSE 0 END) * 100, 1) AS pct_sucesso,
  ROUND(AVG(CASE WHEN ct.status_contrato = 'Cancelado' THEN 1 ELSE 0 END) * 100, 1) AS pct_cancelado
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}contratos` ct
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}clientes` c
  ON ct.cliente_id = c.cliente_id
GROUP BY 1,2,3,4,5
ORDER BY qtd_contratos DESC;


In [0]:
%sql
-- Clientes com maior potencial de recuperação
SELECT
  c.cliente_id,
  c.nome,
  c.score_credito,
  c.renda_mensal,
  c.segmento,
  p.potencial_recuperacao,
  COUNT(*) AS parcelas_em_aberto,
  ROUND(SUM(p.valor_parcela), 2) AS valor_em_atraso,
  MAX(p.dias_atraso) AS max_dias_atraso
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}parcelas_cobranca` p
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}clientes` c
  ON p.cliente_id = c.cliente_id
WHERE p.status_parcela IN ('Vencida', 'Pendente')
  AND p.potencial_recuperacao IN ('Alto', 'Médio')
GROUP BY 1,2,3,4,5,6
ORDER BY valor_em_atraso DESC
LIMIT 20;


---
## Resumo do Workshop

| Módulo | Habilidade |
|---|---|
| 1 | Upload manual de CSVs e criação de tabelas |
| 2 (opc.) | Subir XLSX, transformar com Genie Code e materializar tabelas |
| 3 (opc.) | Pipeline visual no Lakeflow Designer |
| 4 (opc.) | AI Functions (`ai_gen`, `ai_classify`) |
| 5 | Genie Agents para inadimplência, comunicações, recuperação e CNH |
| 6 | Genie One + **report diário agendado** |
| 7 | AI/BI Dashboard com **2 páginas** |

### Próximos passos
1. Troque o dataset sintético pelos dados reais do seu time (com mascaramento adequado)
2. Configure um Genie Space permanente para Cobrança / Risco / CX
3. Mantenha o report diário e evolua os KPIs com o negócio

Documentação: [docs.databricks.com](https://docs.databricks.com)
